# Predicting Chess Outcomes: A Disciplined ML Walkthrough

The full code of [Predicting Chess Outcomes: A Disciplined ML Walkthrough](https://residual-stream.lvtan71.workers.dev/posts/disciplined-ml-chess-outcomes/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/disciplined-ml-chess-outcomes) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/disciplined-ml-chess-outcomes.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 4 min end to end on the RTX 3090 machine that saved the outputs below.
engine.py needs a Stockfish binary; without one it fails, and engine_ab.py uses the folder's committed engine.parquet instead.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 4 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q huggingface_hub pandas pyarrow chess scikit-learn

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "disciplined-ml-chess-outcomes"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/disciplined-ml-chess-outcomes


## `data.py`

Downloads January 2015 from the Lichess database (a pinned parquet copy on Hugging Face), keeps the first 100,000 rated blitz games, and audits them. Writes `games.parquet`.

In [2]:
!{sys.executable} data.py

January 2015: 1,497,237 rated games, 648,250 blitz; kept the first 100,000


they span 2014-12-31 23:00:06 to 2015-01-05 21:02:21 UTC: 4.9 of the month's 31 days
results: White wins 50.0%, draws 3.8%, Black wins 46.2%; White's mean score 0.519
ratings missing: 0

how games ended, and how many moves they lasted
     termination   games  median moves  no moves  under 3 moves
       Abandoned   1,891             0       990         100.0%
          Normal  70,079            31       740           2.1%
Rules infraction       6             5         0           0.0%


    Time forfeit  28,024            39        25           2.0%
abandoned games: White's mean score 0.476, decisive 100.0%, moves played 901
9,626 players; the 1,000 most active play in 76% of games
games reaching move 3: 96.1%
games reaching move 10: 93.7%


The post printed:

```text
January 2015: 1,497,237 rated games, 648,250 blitz; kept the first 100,000
they span 2014-12-31 23:00:06 to 2015-01-05 21:02:21 UTC: 4.9 of the month's 31 days
results: White wins 50.0%, draws 3.8%, Black wins 46.2%; White's mean score 0.519
ratings missing: 0

how games ended, and how many moves they lasted
     termination   games  median moves  no moves  under 3 moves
       Abandoned   1,891             0       990         100.0%
          Normal  70,079            31       740           2.1%
Rules infraction       6             5         0           0.0%
    Time forfeit  28,024            39        25           2.0%
abandoned games: White's mean score 0.476, decisive 100.0%, moves played 901
9,626 players; the 1,000 most active play in 76% of games
games reaching move 3: 96.1%
games reaching move 10: 93.7%
```

## `features.py`

Replays the first ten moves of each game with python-chess and records ratings, time control and board features after moves 3 and 10. Writes `features.parquet`.

In [3]:
!{sys.executable} features.py

100,000 games, 37 columns


games with a move by each side (2+ plies): 96.9%; abandoned among those: 0
move-3 features present: 96.0%; move-10: 93.4%
first moves: e4 59.2%, d4 25.0%, other 3.3%, c4 3.2%, Nf3 2.7%


The post printed:

```text
100,000 games, 37 columns
games with a move by each side (2+ plies): 96.9%; abandoned among those: 0
move-3 features present: 96.0%; move-10: 93.4%
first moves: e4 59.2%, d4 25.0%, other 3.3%, c4 3.2%, Nf3 2.7%
```

## `test_contract.py`

The contract's rules as tests.

In [4]:
!{sys.executable} test_contract.py

ok  test_elo_model_sees_no_rating
ok  test_engine_sees_only_the_move10_board
ok  test_horizons_only_add_information
ok  test_move3_sees_nothing_after_move3
ok  test_move_groups_are_disjoint_from_ratings
ok  test_no_model_sees_its_target
ok  test_no_outcome_or_identity_columns
ok  test_pregame_sees_no_moves
8 rules hold


The post printed:

```text
ok  test_elo_model_sees_no_rating
ok  test_engine_sees_only_the_move10_board
ok  test_horizons_only_add_information
ok  test_move3_sees_nothing_after_move3
ok  test_move_groups_are_disjoint_from_ratings
ok  test_no_model_sees_its_target
ok  test_no_outcome_or_identity_columns
ok  test_pregame_sees_no_moves
8 rules hold
```

## `train.py`

Trains the models on one 80/20 split and scores them against baselines, with a bootstrap interval.

In [5]:
!{sys.executable} train.py

split: 80,079 training and 19,921 validation games; each model then keeps the played games that reach its horizon

model         games Brier: mean     Elo Elo refit   model  gain over Elo refit (95% CI)    ECE


win_pregame  19,304      0.2391  0.2053    0.2043  0.2048      -0.0005 (-0.0008, -0.0003)  0.007


win_move3    19,107      0.2390  0.2051    0.2041  0.2045      -0.0005 (-0.0008, -0.0001)  0.010


win_move10   18,585      0.2388  0.2054    0.2044  0.2025      +0.0018 (+0.0011, +0.0026)  0.007

ratings from play: mean absolute error in rating points, both players


  training mean             187.1   (18,585 games)
  elo_move10                162.1   correlation of the pair's mean rating 0.53


  elo_time_only             173.9   correlation of the pair's mean rating 0.39


  elo_moves_only            174.5   correlation of the pair's mean rating 0.38


The post printed:

```text
split: 80,079 training and 19,921 validation games; each model then keeps the played games that reach its horizon

model         games Brier: mean     Elo Elo refit   model  gain over Elo refit (95% CI)    ECE
win_pregame  19,304      0.2391  0.2053    0.2043  0.2048      -0.0005 (-0.0008, -0.0003)  0.007
win_move3    19,107      0.2390  0.2051    0.2041  0.2045      -0.0005 (-0.0008, -0.0001)  0.010
win_move10   18,585      0.2388  0.2054    0.2044  0.2025      +0.0018 (+0.0011, +0.0026)  0.007

ratings from play: mean absolute error in rating points, both players
  training mean             187.1   (18,585 games)
  elo_move10                162.1   correlation of the pair's mean rating 0.53
  elo_time_only             173.9   correlation of the pair's mean rating 0.39
  elo_moves_only            174.5   correlation of the pair's mean rating 0.38
```

## `seeds.py`

The same on five random splits, a split by time, and a split that keeps each player on one side.

In [6]:
!{sys.executable} seeds.py

       split   train  valid   pregame    move 3   move 10  rating MAE: mean -> model


   random 42  80,079 19,921   -0.0004   -0.0002   +0.0019             187.1 -> 162.0


    random 1  80,045 19,955   -0.0001   -0.0001   +0.0027             187.6 -> 162.9


    random 2  80,019 19,981   -0.0001   -0.0001   +0.0022             188.0 -> 162.5


    random 3  79,738 20,262   -0.0002   -0.0002   +0.0025             188.9 -> 162.6


    random 4  79,980 20,020   -0.0005   -0.0004   +0.0022             187.6 -> 163.1
 random mean                  -0.0003   -0.0002   +0.0023


        time  80,000 20,000   +0.0001   -0.0000   +0.0027             189.2 -> 164.0


     players  43,050 11,606   -0.0011   -0.0009   +0.0015             179.1 -> 164.6


The post printed:

```text
       split   train  valid   pregame    move 3   move 10  rating MAE: mean -> model
   random 42  80,079 19,921   -0.0004   -0.0002   +0.0019             187.1 -> 162.0
    random 1  80,045 19,955   -0.0001   -0.0001   +0.0027             187.6 -> 162.9
    random 2  80,019 19,981   -0.0001   -0.0001   +0.0022             188.0 -> 162.5
    random 3  79,738 20,262   -0.0002   -0.0002   +0.0025             188.9 -> 162.6
    random 4  79,980 20,020   -0.0005   -0.0004   +0.0022             187.6 -> 163.1
 random mean                  -0.0003   -0.0002   +0.0023
        time  80,000 20,000   +0.0001   -0.0000   +0.0027             189.2 -> 164.0
     players  43,050 11,606   -0.0011   -0.0009   +0.0015             179.1 -> 164.6
```

## `leak.py`

Adds one leaked column to the Elo model, shows the score it would report, and the test that catches it.

In [7]:
!{sys.executable} leak.py

test_elo_model_sees_no_rating: FAILED on the edited contract


rating model as contracted  MAE  162.1 rating points
rating model with elo_diff  MAE  143.1 rating points


The post printed:

```text
test_elo_model_sees_no_rating: FAILED on the edited contract
rating model as contracted  MAE  162.1 rating points
rating model with elo_diff  MAE  143.1 rating points
```

## `diagnose.py`

A win-rate table by rating gap and material, a castling check, a learning curve, permutation importance and error by segment.

In [8]:
!{sys.executable} diagnose.py

White's mean score by rating gap (rows) and material after move 10 (columns), training games
mat         <=-2    -1     0    +1  >=+2
gap                                     
Black +150  0.16  0.19  0.25  0.32  0.43
Black +50   0.34  0.38  0.41  0.48  0.57
even        0.45  0.49  0.52  0.57  0.70
White +50   0.54  0.61  0.62  0.68  0.81
White +150  0.72  0.75  0.79  0.84  0.91
games per cell:
mat         <=-2    -1      0    +1  >=+2
gap                                      
Black +150  1958  1636   7993   486   284
Black +50   1709  1618   9459   683   409
even        2432  2376  14418  1236   707
White +50   1543  1562   9655   945   600
White +150  1354  1205   8515  1060   982



even ratings (|gap| <= 50): White's mean score by who had castled by move 10
                              mean  size
m10_castled_w m10_castled_b             
0.0           0.0            0.517  6536
              1.0            0.501  3587
1.0           0.0            0.527  4789
              1.0            0.524  6454

learning curve, win_move10: validation Brier (Elo refit on the full training set: 0.2044)
   games  mean of 3 subsamples (lowest, highest)


   2,500  0.2105 (0.2095, 0.2121)


   5,000  0.2070 (0.2067, 0.2075)


  10,000  0.2051 (0.2045, 0.2057)


  20,000  0.2038 (0.2036, 0.2041)


  40,000  0.2031 (0.2030, 0.2033)


  74,825  0.2026 (0.2025, 0.2027)



permutation importance (Brier increase when shuffled; share of the total)
  elo_diff               0.05950  90.0%
  m10_material_diff      0.00451   6.8%
  m10_developed_b        0.00046   0.7%
  elo_b                  0.00032   0.5%
  elo_mean               0.00025   0.4%
  m10_castled_w          0.00017   0.3%
  m10_developed_w        0.00016   0.2%
  m10_castled_b          0.00012   0.2%

validation Brier of the move-10 model, by |rating gap|


                  mean  size
gap                         
[0.0, 50.0)     0.2335  5081
[50.0, 150.0)   0.2238  7126
[150.0, 300.0)  0.1847  4188
[300.0, inf)    0.0956  2190
by material after move 10 (even vs not)
                     mean   size
m10_material_diff               
uneven             0.1951   6122
even               0.2062  12463


The post printed:

```text
White's mean score by rating gap (rows) and material after move 10 (columns), training games
mat         <=-2    -1     0    +1  >=+2
gap                                     
Black +150  0.16  0.19  0.25  0.32  0.43
Black +50   0.34  0.38  0.41  0.48  0.57
even        0.45  0.49  0.52  0.57  0.70
White +50   0.54  0.61  0.62  0.68  0.81
White +150  0.72  0.75  0.79  0.84  0.91
games per cell:
mat         <=-2    -1      0    +1  >=+2
gap                                      
Black +150  1958  1636   7993   486   284
Black +50   1709  1618   9459   683   409
even        2432  2376  14418  1236   707
White +50   1543  1562   9655   945   600
White +150  1354  1205   8515  1060   982

even ratings (|gap| <= 50): White's mean score by who had castled by move 10
                              mean  size
m10_castled_w m10_castled_b             
0.0           0.0            0.517  6536
              1.0            0.501  3587
1.0           0.0            0.527  4789
              1.0            0.524  6454

learning curve, win_move10: validation Brier (Elo refit on the full training set: 0.2044)
   games  mean of 3 subsamples (lowest, highest)
   2,500  0.2105 (0.2095, 0.2121)
   5,000  0.2070 (0.2067, 0.2075)
  10,000  0.2051 (0.2045, 0.2057)
  20,000  0.2038 (0.2036, 0.2041)
  40,000  0.2031 (0.2030, 0.2033)
  74,825  0.2026 (0.2025, 0.2027)

permutation importance (Brier increase when shuffled; share of the total)
  elo_diff               0.05950  90.0%
  m10_material_diff      0.00451   6.8%
  m10_developed_b        0.00046   0.7%
  elo_b                  0.00032   0.5%
  elo_mean               0.00025   0.4%
  m10_castled_w          0.00017   0.3%
  m10_developed_w        0.00016   0.2%
  m10_castled_b          0.00012   0.2%

validation Brier of the move-10 model, by |rating gap|
                  mean  size
gap                         
[0.0, 50.0)     0.2335  5081
[50.0, 150.0)   0.2238  7126
[150.0, 300.0)  0.1847  4188
[300.0, inf)    0.0956  2190
by material after move 10 (even vs not)
                     mean   size
m10_material_diff               
uneven             0.1951   6122
even               0.2062  12463
```

## `engine.py`

Stockfish's evaluation of every move-10 position. Writes `engine.parquet`.

**Note:** Optional: needs Stockfish.

In [9]:
!{sys.executable} engine.py

multiprocessing.pool.RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
                    ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/pool.py", line 48, in mapstar
    return list(map(*args))
           ^^^^^^^^^^^^^^^^
  File "residual-stream-code/posts/disciplined-ml-chess-outcomes/engine.py", line 21, in evaluate
    engine = chess.engine.SimpleEngine.popen_uci(shutil.which("stockfish"))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/venv/lib/python3.12/site-packages/chess/engine.py", line 3052, in popen_uci
    return cls.popen(UciProtocol, command, timeout=timeout, debug=debug, setpgrp=setpgrp, **popen_args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/venv/lib/python3.12/site-packages/chess/engine.py", line 3044, in popen
   

The post printed:

```text
Stockfish 18, 20,000 nodes per position, 93,410 positions
evaluation after move 10, White's view: median +45 cp, |eval| > 200 cp in 22.2%
```

## `engine_ab.py`

The move-10 model with and without the engine's evaluation, on the same five splits.

In [10]:
!{sys.executable} engine_ab.py

split 42, 18,585 validation games: Brier Elo refit 0.2044, move-10 model 0.2025, with engine 0.1965 (ECE 0.007)


engine column's gain over the same model without it: +0.0060 (95% CI +0.0049, +0.0072)
  material even   12,463 games: Brier 0.2063 -> 0.2027
  material uneven  6,122 games: Brier 0.1948 -> 0.1838



split seed  gain over Elo refit: without  with engine  ratio
        42                       +0.0019      +0.0079   4.2x
         1                       +0.0027      +0.0089   3.3x
         2                       +0.0022      +0.0082   3.7x
         3                       +0.0025      +0.0083   3.3x
         4                       +0.0022      +0.0086   3.9x
      mean                       +0.0023      +0.0084   3.6x


The post printed:

```text
split 42, 18,585 validation games: Brier Elo refit 0.2044, move-10 model 0.2025, with engine 0.1965 (ECE 0.007)
engine column's gain over the same model without it: +0.0060 (95% CI +0.0049, +0.0072)
  material even   12,463 games: Brier 0.2063 -> 0.2027
  material uneven  6,122 games: Brier 0.1948 -> 0.1838

split seed  gain over Elo refit: without  with engine  ratio
        42                       +0.0019      +0.0079   4.2x
         1                       +0.0027      +0.0089   3.3x
         2                       +0.0022      +0.0082   3.7x
         3                       +0.0025      +0.0083   3.3x
         4                       +0.0022      +0.0086   3.9x
      mean                       +0.0023      +0.0084   3.6x
```